# Model Preparation : Decision Tree

Veuillez tester la préparation du modèle sur chaque fichier CSV nettoyé en amont et comparer les résultats obtenus.

### Librairies requises

In [ ]:
!pip install scikit-learn
!pip install pandas

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.dummy import DummyClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import joblib, os

## Préparation du modèle

In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=42)

## Préparation de données pour entraînement

In [ ]:
path = input("Entrer le chemin du dataset CSV pour l'entrainement: ")

if not path.endswith('.csv') or not path:
    path = "../data/movies_cleaned.csv"

In [ ]:
def prepare_data(file_path, targeted_feature="genre_encoded", test_size=0.2, random_state=42):
    dataframe = pd.read_csv(file_path)
    X = dataframe.drop(targeted_feature, axis=1)
    y = dataframe[targeted_feature]
    return train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(path, targeted_feature="genre_encoded")

## Entrainement du modèle

In [ ]:
decision_tree_model.fit(X_train, y_train)

## Test du modèle

In [ ]:
y_pred = decision_tree_model.predict(X_test)
y_pred

- score() → accuracy (précision) du modèle

In [ ]:
score_dt = decision_tree_model.score(X_test, y_test)
score_dt = round(score_dt * 100, 2)
print(f"Accuracy: {score_dt}%")

- f1-score → mesure de la précision du modèle en prenant en compte à la fois la précision et le rappel

In [ ]:
F1_score_dt = classification_report(y_test, y_pred, output_dict=True)
F1_score_dt = F1_score_dt['weighted avg']['f1-score']
print(f"F1-score: {F1_score_dt:.2f}")

- recall → mesure de la capacité du modèle à identifier correctement les classes positives

In [ ]:
recall_dt = classification_report(y_test, y_pred, output_dict=True)
recall_dt = recall_dt['weighted avg']['recall']
print(f"Recall: {recall_dt:.2f}")

- matrice de confusion → table pour comparer les prédictions du modèle avec les vraies étiquettes

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :\n", cm)

- Rapport métriques par classe → précision, rappel et F1-score pour chaque classe

In [ ]:
report = classification_report(y_test, y_pred)
report

## Débuggage du modèle

In [ ]:
# A. Train/Test split correct ?
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("y_train value_counts():\n", pd.Series(y_train).value_counts())
print("y_test value_counts():\n", pd.Series(y_test).value_counts())

# B. Baseline naïve = majority class
majority_class = pd.Series(y_train).mode()[0]
baseline_score = (y_test == majority_class).mean()
print(f"Baseline majority class: {baseline_score:.3f}")

In [ ]:
# A. Features nulles ou constantes ?
print("Features variance:")
print(pd.DataFrame(X_train).var().sort_values().head(10))

# B. Features range
print("Features range:")
print(pd.DataFrame(X_train).describe().loc[['min', 'max']].T)

# C. Features binaires/catégoriques encodées en continu ?
print("Features très concentrées:")
print((pd.DataFrame(X_train).apply(lambda x: len(x.unique())) < 10).sum())

In [ ]:
# D. Importance des features (spécifique Decision Tree)
feature_importances = pd.Series(
    decision_tree_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
print("Importance des features :")
print(feature_importances)

In [ ]:
# E. Profondeur et nombre de feuilles de l'arbre
print(f"Profondeur de l'arbre : {decision_tree_model.get_depth()}")
print(f"Nombre de feuilles    : {decision_tree_model.get_n_leaves()}")

In [ ]:
y_pred = decision_tree_model.predict(X_test)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
cm

- test : Decision Tree avec profondeur limitée (max_depth=10) pour réduire le surapprentissage

In [ ]:
dt_limited = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_limited.fit(X_train, y_train)
score_dt_limited = dt_limited.score(X_test, y_test) * 100
print(f"Decision Tree (max_depth=10): {score_dt_limited:.2f}%")
print(f"Profondeur : {dt_limited.get_depth()} | Feuilles : {dt_limited.get_n_leaves()}")

- test : dummy classifier (random)

In [ ]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
print("Dummy baseline:", dummy_model.score(X_test, y_test))
score_dummy = dummy_model.score(X_test, y_test) * 100

In [ ]:
print(f"Decision Tree (sans limite)   : {score_dt:.2f}%")
print(f"Decision Tree (max_depth=10)  : {score_dt_limited:.2f}%")
print(f"Dummy baseline                : {score_dummy:.2f}%")

- Comparaison des 3 modèles

In [ ]:
if score_dt_limited > score_dt and score_dt_limited > score_dummy:
    print("Le Decision Tree avec max_depth=10 est le meilleur modèle.")
elif score_dt > score_dt_limited and score_dt > score_dummy:
    print("Le Decision Tree sans limite de profondeur est le meilleur modèle.")
else:
    print("Le baseline Dummy est le meilleur modèle.")

---
## Modèle amélioré : TF-IDF (description) + features numériques + Decision Tree

**Pourquoi cette approche ?**
- La colonne `description` est la feature la plus discriminante pour prédire le genre d'un film
- `TF-IDF` convertit le texte en vecteur numérique (fréquence des mots pondérée)
- `DecisionTreeClassifier` peut combiner features textuelles et numériques via un `ColumnTransformer`
- La profondeur est limitée (`max_depth=20`) pour éviter le surapprentissage sur les données TF-IDF (très nombreuses features)

### Chargement du dataset enrichi (avec description)

In [ ]:
path_text = "../data/movies_cleaned4.csv"  # dataset avec description + features numériques
df_text = pd.read_csv(path_text)

print("Shape:", df_text.shape)
print("Colonnes:", df_text.columns.tolist())
print("\nDistribution des genres:")
print(df_text['genre_encoded'].value_counts())
df_text.head(3)

### Échantillonnage stratifié (optionnel — pour accélérer l'entraînement sur grands datasets)

In [ ]:
# Échantillonnage stratifié : max 5000 exemples par genre pour équilibrer les classes
# Mettre SAMPLE = False pour entraîner sur tout le dataset (plus long mais meilleure accuracy)
SAMPLE = True
MAX_PER_CLASS = 5000

if SAMPLE:
    sampled_groups = [
        group.sample(min(len(group), MAX_PER_CLASS), random_state=42)
        for _, group in df_text.groupby('genre_encoded')
    ]
    df_text = pd.concat(sampled_groups, ignore_index=True)
    print(f"Dataset échantillonné : {len(df_text)} lignes")
    print(df_text['genre_encoded'].value_counts())
else:
    print(f"Dataset complet : {len(df_text)} lignes")

### Construction du pipeline TF-IDF + features numériques + Decision Tree

In [ ]:
NUMERIC_FEATURES = ["rating", "votes", "duration_min", "year_num", "certificate_encoded", "popularity"]
TEXT_FEATURE = "description"
TARGET = "genre_encoded"

X_text = df_text.drop(columns=[TARGET])
y_text = df_text[TARGET]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

print("Train:", X_train_t.shape, "| Test:", X_test_t.shape)

In [ ]:
# ColumnTransformer : TF-IDF sur la description + StandardScaler sur les features numériques
# Decision Tree n'exige pas de mise à l'échelle, mais StandardScaler n'impacte pas négativement
preprocessor = ColumnTransformer(transformers=[
    ('tfidf', TfidfVectorizer(
        max_features=10000,   # top 10 000 mots les plus fréquents
        ngram_range=(1, 2),   # unigrammes + bigrammes
        sublinear_tf=True,    # log(tf) pour atténuer les fréquences élevées
        min_df=5,             # ignorer les mots rares (< 5 occurrences)
        stop_words='english'
    ), TEXT_FEATURE),
    ('num', StandardScaler(), NUMERIC_FEATURES)
])

# Pipeline complet : prétraitement + Decision Tree
# max_depth=20 : limite la profondeur pour éviter le surapprentissage sur les features TF-IDF
pipeline_tfidf = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', DecisionTreeClassifier(max_depth=20, random_state=42))
])

pipeline_tfidf.fit(X_train_t, y_train_t)
print("Modèle entraîné.")

### Évaluation du modèle amélioré

In [ ]:
y_pred_tfidf = pipeline_tfidf.predict(X_test_t)

score_tfidf = pipeline_tfidf.score(X_test_t, y_test_t) * 100
print(f"Accuracy (TF-IDF + Decision Tree) : {score_tfidf:.2f}%")

report_tfidf = classification_report(y_test_t, y_pred_tfidf)
print("\nRapport de classification :")
print(report_tfidf)

### Comparaison finale des modèles

In [ ]:
print("=" * 58)
print(f"{'Modèle':<43} {'Accuracy':>10}")
print("=" * 58)
print(f"{'Decision Tree (sans limite)':<43} {score_dt:>9.2f}%")
print(f"{'Decision Tree (max_depth=10)':<43} {score_dt_limited:>9.2f}%")
print(f"{'Dummy (classe majoritaire)':<43} {score_dummy:>9.2f}%")
print(f"{'TF-IDF + Decision Tree (description + num.)':<43} {score_tfidf:>9.2f}%")
print("=" * 58)

best = max(score_dt, score_dt_limited, score_dummy, score_tfidf)
if best == score_tfidf:
    print("\n→ Le modèle TF-IDF + Decision Tree est le meilleur.")
elif best == score_dt_limited:
    print("\n→ Le Decision Tree avec max_depth=10 est le meilleur.")
elif best == score_dt:
    print("\n→ Le Decision Tree sans limite de profondeur est le meilleur.")
else:
    print("\n→ Le baseline Dummy est le meilleur (les features ne sont pas discriminantes).")

## Exportation du modèle

In [ ]:
PATH_OUTPUT_MODEL = "../model"
if not os.path.exists(PATH_OUTPUT_MODEL):
    os.makedirs(PATH_OUTPUT_MODEL, exist_ok=True)

joblib.dump(decision_tree_model, f"{PATH_OUTPUT_MODEL}/decision_tree_model.pkl")
joblib.dump(dt_limited, f"{PATH_OUTPUT_MODEL}/decision_tree_limited_model.pkl")
joblib.dump(dummy_model, f"{PATH_OUTPUT_MODEL}/dummy_model_dt.pkl")
joblib.dump(pipeline_tfidf, f"{PATH_OUTPUT_MODEL}/pipeline_tfidf_dt_model.pkl")

print("Modèles exportés :")
print("  decision_tree_model.pkl         → Decision Tree (sans limite)")
print("  decision_tree_limited_model.pkl → Decision Tree (max_depth=10)")
print("  dummy_model_dt.pkl              → Dummy baseline")
print("  pipeline_tfidf_dt_model.pkl     → TF-IDF + Decision Tree (meilleur)")

# Ressources

- [Decision Tree - scikit-learn](https://scikit-learn.org/stable/modules/tree.html)